In [2]:

# ============================================================
# AI YOUTUBE SCRIPT STUDIO - GOOGLE COLAB SINGLE-CELL SETUP
# Gemini + LangChain + Wikipedia + Streamlit + ngrok
# ============================================================

import os
import sys
import time
import subprocess
from getpass import getpass

# 1. INSTALL DEPENDENCIES
print("Installing dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "streamlit", "langchain-core",
    "langchain-google-genai", "langchain-community",
    "wikipedia", "pyngrok"
], check=True)

# 2. COLLECT API CREDENTIALS
os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")
NGROK_TOKEN = getpass("Enter ngrok auth token: ")

if not os.environ["GOOGLE_API_KEY"].strip() or not NGROK_TOKEN.strip():
    raise ValueError("Both API key and ngrok token are required.")

# 3. CREATE STREAMLIT APPLICATION
app_code = r'''
import os
import streamlit as st
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import WikipediaAPIWrapper

st.set_page_config(
    page_title="Creator Studio | AI",
    page_icon="🎬",
    layout="wide"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&display=swap');
html, body, [class*="css"] {font-family:'DM Sans',sans-serif}
.stApp {background:#F5F7FB}
[data-testid="stSidebar"] {background:#111827}
[data-testid="stSidebar"] * {color:#F3F4F6}
.hero {
 background:linear-gradient(120deg,#172554,#3730A3,#4F46E5);
 padding:32px;border-radius:18px;color:white;margin-bottom:24px
}
.hero h1 {color:white;font-size:34px}
.hero p {color:#E0E7FF}
div.stButton>button[kind="primary"] {
 background:#4F46E5;border:0;border-radius:9px;min-height:46px
}
[data-testid="stMetric"] {
 background:white;border:1px solid #E5E7EB;
 padding:16px;border-radius:12px
}
</style>
""", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("# 🎬 Creator Studio")
    st.caption("AI-powered content workspace")
    st.divider()
    st.markdown("### AI Configuration")
    model_name = st.selectbox(
        "Gemini model",
        ["gemini-3.5-flash", "gemini-3.6-flash", "gemini-3.8-flash"]
    )
    st.caption("Gemini · LangChain · Wikipedia")
    st.divider()
    st.info("Create research-informed video content.")

st.markdown("""
<div class="hero">
<h1>AI YouTube Script Studio</h1>
<p>Transform ideas into engaging, research-informed video content.</p>
</div>
""", unsafe_allow_html=True)

c1, c2, c3 = st.columns(3)
c1.metric("AI Engine", "Gemini")
c2.metric("Research Source", "Wikipedia")
c3.metric("Workflow", "Title → Script")

st.markdown("### Create your next video")

with st.form("generator"):
    topic = st.text_area(
        "Video topic",
        placeholder="e.g. How agentic AI is changing software development",
        height=100
    )

    a, b = st.columns(2)
    with a:
        tone = st.selectbox(
            "Narration style",
            ["Professional", "Conversational", "Documentary",
             "Educational", "Storytelling", "Energetic"]
        )
        audience = st.selectbox(
            "Target audience",
            ["Beginners", "Students", "Technology professionals",
             "General audience", "Advanced learners"]
        )

    with b:
        duration = st.selectbox(
            "Video duration",
            ["1–2 minutes", "3–5 minutes", "5–8 minutes", "8–12 minutes"],
            index=1
        )
        language = st.selectbox("Language", ["English", "Telugu", "Hindi"])

    use_wikipedia = st.checkbox("Include Wikipedia research", value=True)
    generate = st.form_submit_button(
        "✨ Generate Video Content",
        type="primary",
        use_container_width=True
    )

if generate:
    if not topic.strip():
        st.warning("Enter a video topic first.")
    elif not os.getenv("GOOGLE_API_KEY"):
        st.error("Gemini API key is missing.")
    else:
        try:
            with st.spinner("Researching and generating your content..."):
                llm = ChatGoogleGenerativeAI(
                    model=model_name,
                    google_api_key=os.environ["GOOGLE_API_KEY"],
                    temperature=0.7,
                    max_retries=2
                )

                research = "No external research requested."
                if use_wikipedia:
                    try:
                        wiki = WikipediaAPIWrapper(
                            top_k_results=3,
                            doc_content_chars_max=5000
                        )
                        research = wiki.run(topic)
                        if not research.strip():
                            research = "No relevant Wikipedia results found."
                    except Exception:
                        research = (
                            "Wikipedia retrieval failed. "
                            "Avoid unsupported factual claims."
                        )
                        st.warning("Wikipedia research was unavailable.")

                title_prompt = ChatPromptTemplate.from_template("""
You are a professional YouTube content strategist.
Create one compelling and accurate title.
Topic: {topic}
Style: {tone}
Audience: {audience}
Language: {language}
Avoid misleading clickbait. Return only the title.
""")

                title_response = (title_prompt | llm).invoke({
                    "topic": topic, "tone": tone,
                    "audience": audience, "language": language
                })
                title = title_response.content
                if isinstance(title, list):
                    title = "\n".join(str(x) for x in title)
                title = str(title).strip()

                script_prompt = ChatPromptTemplate.from_template("""
You are an experienced YouTube scriptwriter.

TOPIC: {topic}
TITLE: {title}
STYLE: {tone}
AUDIENCE: {audience}
DURATION: {duration}
LANGUAGE: {language}

REFERENCE RESEARCH:
{research}

Write a production-ready Markdown script with:
1. A strong opening hook
2. Introduction and explanation of why the topic matters
3. Structured main sections with clear transitions
4. Suggested B-roll and on-screen visuals in brackets
5. Conclusion and call to action
6. Editing and production notes

Match the requested duration. Use research where relevant.
Never fabricate statistics, quotes, or sources.
Use the requested language throughout.
""")

                script_response = (script_prompt | llm).invoke({
                    "topic": topic, "title": title,
                    "tone": tone, "audience": audience,
                    "duration": duration, "language": language,
                    "research": research
                })
                script = script_response.content
                if isinstance(script, list):
                    script = "\n".join(str(x) for x in script)
                script = str(script).strip()

                st.session_state["title"] = title
                st.session_state["script"] = script
                st.session_state["research"] = research

        except Exception as e:
            st.error("Generation failed. Check API access, quota, and model.")
            st.code(str(e))

if "script" in st.session_state:
    st.divider()
    st.markdown("## Generated Content")

    t1, t2, t3 = st.tabs(["📝 Video Title", "🎥 Full Script", "🔎 Research"])

    with t1:
        st.subheader(st.session_state["title"])
        st.code(st.session_state["title"])
        st.download_button(
            "Download Title",
            st.session_state["title"],
            file_name="youtube_title.txt"
        )

    with t2:
        st.markdown(st.session_state["script"])
        st.download_button(
            "⬇ Download Complete Script",
            st.session_state["script"],
            file_name="youtube_script.md",
            mime="text/markdown",
            type="primary"
        )

    with t3:
        st.text_area(
            "Wikipedia research",
            st.session_state["research"],
            height=350
        )

    st.caption("Review factual claims before publishing.")

st.divider()
st.caption("CREATOR STUDIO · AI CONTENT WORKFLOW")
'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

# 4. START STREAMLIT
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

# Clean up an old tunnel if this cell is rerun.
for tunnel in ngrok.get_tunnels():
    try:
        ngrok.disconnect(tunnel.public_url)
    except Exception:
        pass

# Launch the Streamlit server.
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit to become ready.
import urllib.request

ready = False
for _ in range(30):
    if process.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(2)

if not ready:
    print("Streamlit did not start. Recent logs:")
    try:
        print(open("/content/streamlit.log").read()[-5000:])
    except Exception:
        pass
    raise RuntimeError("Streamlit startup failed. Check the logs above.")

# 5. CREATE PUBLIC URL
public_url = ngrok.connect(8501, "http")
print("\n" + "=" * 55)
print("AI YOUTUBE SCRIPT STUDIO IS RUNNING")
print("=" * 55)
print("Open this URL:", public_url.public_url)
print("Keep this Colab cell/runtime running while using the app.")


Installing dependencies...
Enter Gemini API key: ··········
Enter ngrok auth token: ··········

AI YOUTUBE SCRIPT STUDIO IS RUNNING
Open this URL: https://amulet-dinner-lesser.ngrok-free.dev
Keep this Colab cell/runtime running while using the app.
